In [49]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [50]:
data = pd.read_csv('/content/drive/MyDrive/Ratings.csv', sep=',', encoding="latin-1", on_bad_lines='skip')
data_books = pd.read_csv('/content/drive/MyDrive/Books.csv', sep=',', encoding="latin-1", on_bad_lines='skip', low_memory=False)

In [51]:
data.columns = ['userId', 'isbn', 'rating']

In [52]:
data_books.rename(columns={data_books.columns[0]: 'isbn', data_books.columns[1]: 'title'}, inplace=True)

In [53]:
user_counts = data['userId'].value_counts()
book_counts = data['isbn'].value_counts()
data_ratings = data[data['userId'].isin(user_counts[user_counts > 50].index)]
data_ratings = data_ratings[data_ratings['isbn'].isin(book_counts[book_counts > 50].index)]

users = data_ratings['userId'].unique()
books = data_ratings['isbn'].unique()

print("Number of users: ", len(users))
print("Number of books: ", len(books))

Number of users:  3237
Number of books:  2125


In [54]:
ratings_mat = np.zeros(shape=(len(books), len(users)), dtype=np.float32)
print("Matrix Shape:", ratings_mat.shape)

ratings_array = data_ratings.rating.values
isbn_array = data_ratings.isbn.values
userIds_array = data_ratings.userId.values

Matrix Shape: (2125, 3237)


In [55]:
books_dict_ref_real = {idx: res for idx, res in enumerate(books)}
books_dict_real_ref = {res: idx for idx, res in enumerate(books)}
users_dict_real_ref = {res: idx for idx, res in enumerate(users)}

In [56]:
for i in range(len(ratings_array)):
    u = userIds_array[i]
    b = isbn_array[i]

    ref_u = users_dict_real_ref[u]
    ref_b = books_dict_real_ref[b]

    ratings_mat[ref_b, ref_u] = ratings_array[i]

print("Matrix successfully populated!")

Matrix successfully populated!


In [57]:
U, S, V = np.linalg.svd(ratings_mat, full_matrices=False)
print("U shape:", U.shape)
print("S shape:", S.shape)
print("V shape:", V.shape)

U shape: (2125, 2125)
S shape: (2125,)
V shape: (2125, 3237)


In [58]:
def top_cosine_similarity(data, book_id, k, top_n):
    book_row = np.array(data[book_id, :]).reshape(1, -1)

    similarity = cosine_similarity(book_row, data)

    sort_indices = np.argsort(similarity[0])[::-1]

    return sort_indices[1:top_n+1]

In [59]:
k = 50

search_isbn = data_ratings['isbn'].iloc[0]
ref_book_id = books_dict_real_ref[search_isbn]

sliced_matrix = np.array(U[:, :k])

top_indices = top_cosine_similarity(sliced_matrix, ref_book_id, k, top_n=10)

search_title = data_books[data_books.isbn == search_isbn].title.values[0]
print(f"Because you read: '{search_title}'\n" + "="*40)
print("Recommended Books:")

for ref in top_indices:
    real_isbn = books_dict_ref_real[ref]
    try:
        title = data_books[data_books.isbn == real_isbn].title.values[0]
        print(f"- {title}")
    except:
        print(f"- [Title not found for ISBN: {real_isbn}]")

Because you read: 'Along Came a Spider (Alex Cross Novels)'
Recommended Books:
- Term Limits
- The Eleventh Commandment
- When the Wind Blows
- Critical Mass
- Kiss the Girls
- Unfit to Practice
- Jack &amp; Jill (Alex Cross Novels)
- Hide &amp; Seek
- The Program
- Manhattan Hunt Club


In [60]:
print(data_ratings.head())
print("\n")

print(data_books[['isbn', 'title', 'Book-Author']].head())
print("\n")

print(ratings_mat.shape)
print("\n")

print(data_books[['isbn', 'title', 'Book-Author']])

     userId        isbn  rating
173  276847  0446364193       0
182  276847  3426029553       8
413  276925  002542730X      10
426  276925  0316666343       0
427  276925  0345391810       0


         isbn                                              title  \
0  0195153448                                Classical Mythology   
1  0002005018                                       Clara Callan   
2  0060973129                               Decision in Normandy   
3  0374157065  Flu: The Story of the Great Influenza Pandemic...   
4  0393045218                             The Mummies of Urumchi   

            Book-Author  
0    Mark P. O. Morford  
1  Richard Bruce Wright  
2          Carlo D'Este  
3      Gina Bari Kolata  
4       E. J. W. Barber  


(2125, 3237)


              isbn                                              title  \
0       0195153448                                Classical Mythology   
1       0002005018                                       Clara Callan   
2    

In [61]:
print(ratings_mat[:20, :20].astype(int))

print(ratings_mat.shape)

[[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 8  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0 10  0  0  0  0  0 10  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  9  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  8  0  0  0  0  0  8  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  8  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  7  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  6  0  0  5  0  0  0  0  0  0  0]
 [ 0  0  0  6  0  0  0  0  0  0  0  0  0  0  0  0  0  0

In [62]:
k = 50

search_isbn = data_ratings['isbn'].iloc[15]

ref_book_id = books_dict_real_ref[search_isbn]

sliced_matrix = np.array(U[:, :k])

top_indices = top_cosine_similarity(sliced_matrix, ref_book_id, k, top_n=10)

print(top_indices)

[1002  304  620 1862 2081  335 1765  718  719 1298]


In [64]:
k = 200

search_isbn = data_ratings['isbn'].iloc[199]
ref_book_id = books_dict_real_ref[search_isbn]

sliced_matrix = np.array(U[:, :k])

top_indices = top_cosine_similarity(sliced_matrix, ref_book_id, k, top_n=10)

search_title = data_books[data_books.isbn == search_isbn].title.values[0]

print(search_title)
print("recommended books ")

for ref in top_indices:
    real_isbn = books_dict_ref_real[ref]
    print(real_isbn)

    try:
        title = data_books[data_books.isbn == real_isbn].title.values[0]
        print(title)
    except:
        print("[Title not found for this ISBN]")

Midnight in the Garden of Good and Evil
recommended books 
038082101X
Daughter of Fortune: A Novel
0671683993
The Temple of My Familiar
0060740450
One Hundred Years of Solitude (Oprah's Book Club)
074322535X
Gap Creek: The Story of a Marriage
0345465083
Seabiscuit
0449206475
The Witches of Eastwick
0671000306
Shock Wave (Dirk Pitt Adventures (Paperback))
0743410181
Temptation
006109868X
Pigs in Heaven
0671793535
A Rose For Her Grave &amp; Other True Cases (Ann Rule's Crime Files)


In [21]:
'/content/drive/MyDrive/Ratings.csv'

'/content/drive/MyDrive/Ratings.csv'

In [22]:
'/content/drive/MyDrive/Books.csv'

'/content/drive/MyDrive/Books.csv'